In [2]:
import xarray as xr
import numpy as np
from eofs.xarray import Eof

import pandas as pd
import re

from pathlib import Path

- read data and prep

In [10]:
u_file = "processed/era5_uwind_all_lv_monthly_1979_2023.nc" # monthly 

In [11]:
xu_era5 = xr.open_dataset(u_file)

In [13]:
xu_era5.mean(dim='lon')

<xarray.Dataset> Size: 404kB
Dimensions:  (lat: 5, plev: 37, time: 540)
Coordinates:
  * lat      (lat) float64 40B -56.0 -58.0 -60.0 -62.0 -64.0
  * plev     (plev) float64 296B 100.0 200.0 300.0 ... 9.5e+04 9.75e+04 1e+05
  * time     (time) datetime64[ns] 4kB 1979-01-01 1979-02-01 ... 2023-12-01
Data variables:
    uwind    (time, plev, lat) float32 400kB -25.51 -24.54 ... 4.618 2.848

In [118]:
def zonal_area_mean(
    da: xr.DataArray | xr.Dataset,
    lat_band: tuple[float, float] = (-65, -55),
    lat_name: str = "lat",
    lon_name: str = "lon",
) -> xr.DataArray:
    """
    Compute zonal mean and area-weighted mean over a latitude band.
    Automatically sorts latitude to ascending order.

    Parameters
    ----------
    da : xr.DataArray or xr.Dataset
        Input data with dimensions including `lat` and `lon`.
        e.g. uwind(time, plev, lat, lon)
    lat_band : tuple(float, float)
        Latitude range (south, north), e.g. (-65, -55) for 55–65°S.
    lat_name, lon_name : str
        Coordinate names (defaults are 'lat' and 'lon').

    Returns
    -------
    da_mean : xr.DataArray
        Zonal + area-weighted mean, dimensions (time, plev).
    """

    # --- Step 0: ensure latitude ascending (S -> N) ---
    if da[lat_name][0] > da[lat_name][-1]:
        da = da.sortby(lat_name)

    # --- Step 1: take zonal mean ---
    if lon_name in da.dims:
        da_zm = da.mean(lon_name)
    else:
        da_zm = da

    # --- Step 2: select latitude band ---
    latmin, latmax = sorted(lat_band)
    da_sel = da_zm.sel({lat_name: slice(latmin, latmax)})

    # --- Step 3: cosine latitude weights ---
    lat_radians = np.deg2rad(da_sel[lat_name])
    weights = np.cos(lat_radians)
    weights /= weights.sum()  # normalize so they sum to 1

    # --- Step 4: weighted mean over latitude ---
    da_wt = (da_sel * weights).sum(lat_name)

    # add year and month coordinates
    da_wt = da_wt.assign_coords(
        year=da_wt.time.dt.year,
        month=da_wt.time.dt.month
    )

    
    # reshape: create MultiIndex and unstack to get (year, month, plev)
    da_year_month = (
        da_wt
        .set_index(time=('year', 'month'))
        .unstack('time')
        .transpose('year', 'month', 'plev')
    )
    

    return da_year_month


In [119]:
ds = zonal_area_mean(xu_era5)

In [120]:
ds

<xarray.Dataset> Size: 161kB
Dimensions:  (year: 45, month: 12, plev: 37)
Coordinates:
  * year     (year) int64 360B 1979 1980 1981 1982 1983 ... 2020 2021 2022 2023
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * plev     (plev) float64 296B 100.0 200.0 300.0 ... 9.5e+04 9.75e+04 1e+05
Data variables:
    uwind    (year, month, plev) float64 160kB -24.54 -20.39 ... 7.695 6.105

In [181]:
def eof(X,n=-1,detrend='constant',eof_in=None):
	"""Principal Component Analysis / Empirical Orthogonal Functions / SVD

		Uses Singular Value Decomposition to find the dominant modes of variability.
		The field X can be reconstructed with Y = dot(EOF,PC) + X.mean(axis=time)

		INPUTS:
			X	-- Field, shape (time x space).
			n	-- Number of modes to extract. All modes if n < 0
			detrend -- detrend with global mean ('constant')
						  or linear trend ('linear')
		    eof_in  -- If not None, compute PC by projecting eof onto X.
		OUTPUTS:
			EOF - Spatial modes of variability
			PC  - Temporal evolution of EOFs - only output if eof_in is not None
			E   - Explained value of variability
			u   - spatial modes
			s   - variances
			v   - temporal modes

    from aostools from Martin Jucker
	"""
	is_xr = False
	try:
		import xarray as xr
		if isinstance(X,xr.DataArray):
			is_xr = True
			dims = []
			for dim in X.dims[1:]:
				dims.append(X[dim])
			tdim = [X[X.dims[0]]]
			X = X.values
	except:
		pass
	import scipy.signal as sg
	# make sure we have a matrix time x space
	shpe = X.shape
	if len(shpe) > 2:
		X = X.reshape([shpe[0],np.prod(shpe[1:])])
		if eof_in is not None:
			if len(eof_in.shape) > 2:
				eof_in = eof_in.reshape([np.prod(eof_in.shape[:-1]),eof_in.shape[-1]])
			else:
				eof_in = eof_in.reshape([np.prod(eof_in.shape),1])
	# take out the time mean or trend
	X = sg.detrend(X.transpose(),type=detrend)
	if eof_in is not None:
		if eof_in.shape[-1] == X.shape[0]:
			PC =  np.matmul(eof_in, X)
			eof_norm = np.dot(eof_in.transpose(),eof_in)
			return np.dot(PC,np.linalg.inv(eof_norm))
		else:
			PC = np.matmul(eof_in.transpose(), X)
			eof_norm = np.dot(eof_in.transpose(),eof_in)
			return np.dot(PC.transpose(),np.linalg.inv(eof_norm)).transpose()
		# return sg.detrend(PC,type='constant')
	# perform SVD - v is actually V.H in X = U*S*V.H
	u,s,v = np.linalg.svd(X, full_matrices=False)
	# now, u contains the spatial, and v the temporal structures
	# s contains the variances, with the same units as the input X
	# u.shape = (space, modes(space)), v.shape = (modes(space), time)

	# get the first n modes, in physical units
	#  we can either project the data onto the principal component, X*V
	#  or multiply u*s. This is the same, as U*S*V.H*V = U*S
	if n < 0:
		n = s.shape[0]
	EOF = np.dot(u[:,:n],np.diag(s)[:n,:n])
	# time evolution is in v
	PC  = v[:n,:]
	# EOF wants \lambda = the squares of the eigenvalues,
	#  but SVD yields \gamma = \sqrt{\lambda}
	s2 = s*s
	E   = s2[:n]/sum(s2)
	# now we need to make sure we get everything into the correct shape again
	u = u[:,:n]
	s = s[:n]
	v = v.transpose()[:,:n]
	if len(shpe) > 2:
		# replace time dimension with modes at the end of the array
		newshape = list(shpe[1:])+[n]
		EOF = EOF.reshape(newshape)
		u   = u	 .reshape(newshape)
	if is_xr: # return xarray dataarrays
		mode = [('n',np.arange(1,n+1))]
		EOF = xr.DataArray(EOF,coords=dims+mode,name='EOF')
		PC  = xr.DataArray(PC,coords=tdim+mode,name='PC')
		E   = xr.DataArray(E,coords=mode,name='E')
	return EOF,PC,E,u,s,v

In [193]:
def output_stcmI2(ds, outfname, save_txt=True):

    anom = ds

    # Rename 'year' → 'time' so Eof sees it as the sample axis
    anom_year_space = anom.rename(year='time')
    
    EOF,PC,E,u,s,v = eof(anom_year_space)
    eof1 = EOF.sel(n=1)
    pc1 = PC.sel(n=1)

    # -------------------------
    # PC1 time series (standardized & sign-adjusted)
    # -------------------------
    pc1 = pc1 * -1.0
    pc1 = (pc1 - pc1.mean()) / pc1.std()

    if save_txt:
    # -------------------------
    # Save PC1 time series
    # -------------------------
        np.savetxt(outfname, pc1.values)

    return pc1

In [178]:
def output_stcmI(ds, outfname, save_txt=True):

    anom = ds

    # Rename 'year' → 'time' so Eof sees it as the sample axis
    anom_year_space = anom.rename(year='time')
    
    solver = Eof(anom_year_space)
    
    eofs = solver.eofs(neofs=3)        # (mode, space)
    pcs  = solver.pcs(npcs=3, pcscaling=1)  # (time=44, mode)
    varfrac = solver.varianceFraction(neigs=3)

    # -------------------------
    # PC1 time series (standardized & sign-adjusted)
    # -------------------------
    pc1 = pcs[:,0] * -1.0
    pc1 = (pc1 - pc1.mean()) / pc1.std()

    if save_txt:
    # -------------------------
    # Save PC1 time series
    # -------------------------
        np.savetxt(outfname, pc1.values)

    return pc1

In [171]:
output_stcmI(anom, save_txt=True)

<xarray.DataArray 'pcs' (time: 45)> Size: 360B
array([ 0.82120857, -0.17498345, -0.58280701,  0.50654823, -0.3948756 ,
        0.42604543, -0.42532185,  0.3227071 , -1.57224784,  1.51972418,
       -0.58354633, -0.5794885 ,  0.72325301,  0.60175443, -0.37338948,
        0.49262067, -0.59210671,  0.03327636,  0.12486901, -0.88619046,
       -1.07546514,  1.02657367, -1.146714  ,  2.43578519,  0.47946137,
        0.75020005,  0.93989336, -0.85103534,  0.52422393, -0.6906417 ,
        0.01071795, -0.82326729, -0.85136604,  1.33978263,  1.18793443,
        0.38937314, -1.61225938,  0.94749896,  0.72075524, -0.81891881,
        2.51783011, -1.46777596, -1.61504505, -1.38706793, -0.33752315])
Coordinates:
  * time     (time) int64 360B 1979 1980 1981 1982 1983 ... 2020 2021 2022 2023
    mode     int64 8B 0

In [201]:
import os


def compute_stcmI(xu_era5, outdir, save_txt=True):
    """
    Compute the STC Mode Index (STCMI) from ERA5 zonal wind data.

    Steps:
      1. Compute zonal and latitude-band means (creates uwind anomalies).
      2. Compute EOFs of (plev × month) anomalies.
      3. Extract standardized PC1 as STCMI.
      4. Optionally save PC1 as text file named by the actual year range.
      5. Skip computation if file already exists.

    Parameters
    ----------
    xu_era5 : str or xarray.Dataset
        Input ERA5 uwind dataset (time, plev, lat, lon) or path to NetCDF.
    outdir : str
        Directory to save output file (e.g., "results/").
    save_txt : bool, default=True
        If True, saves standardized PC1 time series as "stcmI.<start>-<end>.txt".

    Returns
    -------
    dict
        {
            'pc1'     : xarray.DataArray, standardized leading PC
            'pcs'     : xarray.DataArray, first 3 PCs
            'eofs'    : xarray.DataArray, first 3 EOFs
            'varfrac' : xarray.DataArray, variance fractions
        }
    """

    # -----------------------------
    # 0. Load dataset
    # -----------------------------
    ds = xu_era5 
    
    # Check that 'time' exists
    if 'time' not in ds.coords:
        raise ValueError("Input dataset must have a 'time' coordinate.")

    # -----------------------------
    # 1. Get year range dynamically
    # -----------------------------
    years = ds['time'].dt.year
    start_year = int(years.min())
    end_year = int(years.max())

    # -----------------------------
    # 2. Prepare output directory & file name
    # -----------------------------
    os.makedirs(outdir, exist_ok=True)
    outfile = os.path.join(outdir, f"stcmI2.{start_year}-{end_year}.txt")

    if os.path.exists(outfile):
        print(f" File already exists: {outfile}")
        print("→ Skipping computation. To recompute, delete the file first.")
        return None

    # -----------------------------
    # 3. Compute zonal & lat-band means
    # -----------------------------
    anom = zonal_area_mean(ds)  # returns uwind anomalies (time, plev)

    # -----------------------------
    # 4. Compute EOFs and PC1
    # -----------------------------
    result = output_stcmI2(anom, outfile, save_txt=save_txt)

    print(f" STCMI successfully computed for {start_year}–{end_year}")
    print(f"   Saved to: {outfile}")

    return result


In [202]:
compute_stcmI(xu_era5['uwind'], "processed/", save_txt=True)

 STCMI successfully computed for 1979–2023
   Saved to: processed/stcmI2.1979-2023.txt


<xarray.DataArray 'PC' (time: 45)> Size: 360B
array([ 0.68221659, -0.07987031,  1.86967162, -0.74515854,  0.69276204,
       -0.75204749, -0.13768221, -0.4073831 ,  0.07995741, -1.00156077,
        0.22369531, -0.75688179,  0.53098373, -0.771675  ,  0.65734968,
       -0.69651164,  0.08826419,  0.22895599, -2.08157377,  1.18626145,
        0.29312203,  2.50381181, -1.32839911, -0.54965136,  0.74417776,
       -0.07382617, -0.0371132 , -0.00656851,  0.09413159,  0.40331272,
        0.48062578,  0.50522624,  0.19694752,  0.58655781, -2.77926995,
       -1.32270726, -1.74908428, -0.86670526, -0.72632659, -0.50483604,
        1.02514522,  0.79313663,  1.27090695,  1.37465861,  0.86295367])
Coordinates:
  * time     (time) int64 360B 1979 1980 1981 1982 1983 ... 2020 2021 2022 2023
    n        int64 8B 1

In [174]:
def classify_stmi_events(inFile, thresh=None, save_txt=True, save_mask=True):
    """
    Classify polar vortex events based on STCMI time series following Lim et al. (2018, 2019).
    
    Parameters
    ----------
    inFile : str
        Input file containing STCMI time series (one value per year).
        File name must contain year span, e.g. 'stcmI.1979-2023.txt'.
    thresh : float, optional
        Threshold (in std dev units) for defining events. Default is 0.8.
    save_txt : bool, optional
        If True, save a formatted text file with weak/strong event years. Default True.
    save_mask : bool, optional
        If True, save a CSV mask with year, STMI, and classification. Default True.
    
    Returns
    -------
    stmi : pandas.DataFrame
        DataFrame with columns: ['year','STMI','mask']
        mask is one of {'weak','strong','neutral'}.
    outFile, maskFile : str
        Paths to the saved files (or None if not saved).
    """

    # default
    if thresh is None:
        thresh = 0.8
        
    # --- Parse years from filename ---
    match = re.search(r'(\d{4})-(\d{4})', inFile)
    if not match:
        raise ValueError("Filename must contain year span like '1979-2023'")
    startYear, endYear = map(int, match.groups())

    # --- Read STMI time series ---
    stmi = pd.read_csv(inFile, header=None, names=['STMI'])
    stmi['year'] = np.arange(startYear, startYear+len(stmi))

    # --- Apply classification ---
    stmi['mask'] = 'neutral'
    stmi.loc[stmi['STMI'] >  thresh, 'mask'] = 'weak'
    stmi.loc[stmi['STMI'] < -thresh, 'mask'] = 'strong'


    # --- Save formatted text file ---
    if save_txt:
        if save_txt is True:
            out_dir = Path(".")
        else:
            out_dir = Path(save_txt)
            out_dir.mkdir(parents=True, exist_ok=True)

        # --- Prepare file names ---
        outFile  = out_dir / f"STmode_{startYear}-{endYear}_thr±{thresh:.1f}.txt" 
        

        weak_years   = stmi.loc[stmi['mask'] == 'weak', 'year'].tolist()
        strong_years = stmi.loc[stmi['mask'] == 'strong', 'year'].tolist()
        with open(outFile,'w') as fle:
            fle.write("# Polar Vortex weakening and strengthening events based on the multiple EOF (Lim et al. 2018)\n")
            fle.write(f"# Event threshold ±{thresh} stddev following Lim et al. (2019)\n")
            fle.write("# Weakening years\n")
            for y in weak_years:
                fle.write(f"{y}\n")
            fle.write("#--------------------------\n")
            fle.write("# Strengthening years\n")
            for y in strong_years:
                fle.write(f"{y}\n")
    
    # --- Save mask CSV ---
    if save_mask:
        if save_txt is True:
            out_dir = Path(".")
        else:
            out_dir = Path(save_txt)
            out_dir.mkdir(parents=True, exist_ok=True)
            
        maskFile = out_dir / f"STmode_mask_{startYear}-{endYear}_thr±{thresh:.1f}.csv" 
        stmi[['year','STMI','mask']].to_csv(maskFile, index=False)

    return stmi


In [ ]:
stcmi_f = "processed/stcmI.1979-2023.txt"